In [2]:
from huggingface_hub import login

# Replace 'YOUR_TOKEN_HERE' with your actual Hugging Face token
login(token="")


In [4]:
import os
os.environ["HF_TOKEN"] = ""


In [36]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B"  # Hugging Face model ID

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    low_cpu_mem_usage=True,
    device_map="auto",
    torch_dtype="auto"
)


In [38]:
import pandas as pd
import torch

# Load the file with tab separator and skip bad lines
df = pd.read_csv("FLAT_CMPL.txt", sep="\t", header=None, engine="python", on_bad_lines="skip")

# Rename relevant columns
df = df[[11, 19]]
df.columns = ["COMPDESC", "CDESCR"]



In [40]:
df.head()

,COMPDESC,CDESCR
0,ENGINE AND ENGINE COOLING:COOLING SYSTEM:RADIA...,RADIATOR FAILED @ HIGHWAY SPEED OBSTRUCTING DR...
1,"FUEL SYSTEM, GASOLINE:DELIVERY","FUEL LEAKED FROM FUEL TANK AREA, EMITTING STRO..."
2,POWER TRAIN:AUTOMATIC TRANSMISSION,SHIFTED INTO REVERSE VEHICLE JERKED VIOLENTLY....
3,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY",FUEL TANK ; LEAKS BECAUSE OF RUST GAS LEAK BY ...
4,SEATS,"DRIVER SIDE SEAT FRAME BROKE IN TWO, CAUSING S..."


In [42]:
import pandas as pd

# Your target categories
target_categories = [
    'ELECTRICAL SYSTEM',
    'AIR BAGS',
    'WHEELS',
    'ENGINE AND ENGINE COOLING',
    'STRUCTURE'
]

# Filter df to keep only rows where COMPDESC is in target_categories
filtered_df = df[df['COMPDESC'].isin(target_categories)]

# Display or use the filtered dataframe
print(filtered_df)


                  COMPDESC                                             CDESCR
22                AIR BAGS  UPON IMPACT, DURING FRONTAL CRASH AT APPROX. 1...
65                  WHEELS  RIGHT FRONT WHEEL LOCKED UP VEHICLE WENT INTO ...
154               AIR BAGS  DRIVER AND PASSENGER AIRBAG DID NOT DEPLOY IN ...
228                 WHEELS  WHEEL CRACKED ON RIGHT FRONT WHEEL WHILE DRIVI...
310                 WHEELS                     WHEELS SKID WHEN BRAKING.  *AK
...                    ...                                                ...
1681978             WHEELS  I TOOK MY TRUCK INTO THE DEALERSHIP FOR SERVIC...
1681979           AIR BAGS  I KEEP RECEIVING SAFETY RECALL LETTERS ABOUT T...
1681984  ELECTRICAL SYSTEM  SO FROM THE START I BOUGHT THE CAR $5000 CASH ...
1681992  ELECTRICAL SYSTEM  THE CAR CAUGHT ON FIRE, STARTING IN THE ROOF. ...
1681998           AIR BAGS  I BOUGHT THE CAR IN APRIL.2020, THE AIR BAG WA...

[293318 rows x 2 columns]


In [44]:
filtered_df.head()

,COMPDESC,CDESCR
22,AIR BAGS,"UPON IMPACT, DURING FRONTAL CRASH AT APPROX. 1..."
65,WHEELS,RIGHT FRONT WHEEL LOCKED UP VEHICLE WENT INTO ...
154,AIR BAGS,DRIVER AND PASSENGER AIRBAG DID NOT DEPLOY IN ...
228,WHEELS,WHEEL CRACKED ON RIGHT FRONT WHEEL WHILE DRIVI...
310,WHEELS,WHEELS SKID WHEN BRAKING. *AK


In [46]:
# Define the mapping for your categories to labels
category_to_label = {
    'ELECTRICAL SYSTEM': 1,
    'AIR BAGS': 2,
    'WHEELS': 3,
    'ENGINE AND ENGINE COOLING': 4,
    'STRUCTURE': 5
}

# Apply the mapping to filtered_df to create a new column 'Category_Label'
filtered_df['Category_Label'] = filtered_df['COMPDESC'].map(category_to_label)



/var/folders/hx/q1s4c6ws0dn84_dnnjvs7skc0000gp/T/ipykernel_50873/83180464.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Category_Label'] = filtered_df['COMPDESC'].map(category_to_label)


In [48]:
filtered_df.head()

,COMPDESC,CDESCR,Category_Label
22,AIR BAGS,"UPON IMPACT, DURING FRONTAL CRASH AT APPROX. 1...",2
65,WHEELS,RIGHT FRONT WHEEL LOCKED UP VEHICLE WENT INTO ...,3
154,AIR BAGS,DRIVER AND PASSENGER AIRBAG DID NOT DEPLOY IN ...,2
228,WHEELS,WHEEL CRACKED ON RIGHT FRONT WHEEL WHILE DRIVI...,3
310,WHEELS,WHEELS SKID WHEN BRAKING. *AK,3


In [50]:
filtered_df.shape

(293318, 3)

In [74]:
def classify_complaint_fewshot(text, max_new_tokens=20):
    """
    Robust few-shot classification using LLaMA.
    Returns ONLY the category number (1-5).
    """
    prompt = f"""
Classify car complaints into these categories:

1: ELECTRICAL SYSTEM
2: AIR BAGS
3: WHEELS
4: ENGINE AND ENGINE COOLING
5: STRUCTURE

Examples:
Complaint: "Battery drains overnight and power windows stop working."
Answer: 1

Complaint: "Airbag warning light is on and airbags did not deploy during a collision."
Answer: 2

Complaint: "Front right wheel locked up while driving and tire shows cracks."
Answer: 3

Complaint: "Engine stalls frequently and temperature gauge shows inaccurate readings."
Answer: 4

Complaint: "Rear door will not close properly and body panel is misaligned."
Answer: 5

Classify the following complaint.
Return ONLY the category number (1-5), no text or explanation.

Complaint: "{text}"
Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # Take the first valid digit 1-5
    response = next((c for c in response if c in '12345'), "1")

    return response


In [126]:
def classify_complaint_structured_show(text, max_new_tokens=20):
    """
    Classify a car complaint using a structured prompt for LLaMA.
    Returns ONLY the category number (1-5).
    """
    prompt = f'''Human: You are an expert field quality engineer. 
Classify the car complaint below into one of these categories. 
Return ONLY the category number (1-5). No explanation.

1: ELECTRICAL
2: AIR BAGS
3: WHEELS
4: ENGINE
5: STRUCTURE

Complaint: "{text}"
Assistant:'''

    # Encode the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate output tokens
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,          # greedy decoding
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode generated tokens, skip the prompt part
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # Keep only the first valid digit 1-5
    response = next((c for c in response if c in '12345'), "1")

    return response


In [128]:
complaints = [
    # 1: ELECTRICAL SYSTEM
    "The car battery dies overnight and the headlights do not turn on.",
    
    # 2: AIR BAGS
    "The airbag has been torned",
    
    # 3: WHEELS
    "The right front wheel locks up while driving and the tire has visible cracks.",
    
    # 4: ENGINE AND ENGINE COOLING
    "The engine overheats, the temperature gauge is inaccurate, and the car stalls frequently.",
    
    # 5: STRUCTURE
    "The rear door will not close properly and the roof panel is dented after parking."
]

# Classify each complaint
for c in complaints:
    category = classify_complaint_structured_show(c)
    print(f"Complaint: {c}\nPredicted Category: {category}\n")


/opt/anaconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Complaint: The car battery dies overnight and the headlights do not turn on.
Predicted Category: 1



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Complaint: The airbag has been torned
Predicted Category: 1



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Complaint: The right front wheel locks up while driving and the tire has visible cracks.
Predicted Category: 1



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Complaint: The engine overheats, the temperature gauge is inaccurate, and the car stalls frequently.
Predicted Category: 1

Complaint: The rear door will not close properly and the roof panel is dented after parking.
Predicted Category: 1



In [78]:

sample_df = filtered_df.head(10000).copy()

# Run prediction and keep ground truth
sample_df['predicted_category'] = sample_df['CDESCR'].apply(classify_complaint_fewshot)

# Show sample with complaint, ground truth, and prediction
print(sample_df[['CDESCR', 'Category_Label', 'predicted_category']].head(100))

/opt/anaconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` 

                                                 CDESCR  Category_Label  \
22    UPON IMPACT, DURING FRONTAL CRASH AT APPROX. 1...               2   
65    RIGHT FRONT WHEEL LOCKED UP VEHICLE WENT INTO ...               3   
154   DRIVER AND PASSENGER AIRBAG DID NOT DEPLOY IN ...               2   
228   WHEEL CRACKED ON RIGHT FRONT WHEEL WHILE DRIVI...               3   
310                      WHEELS SKID WHEN BRAKING.  *AK               3   
...                                                 ...             ...   
5961  TEMERATURE GAUGE FAILED, SHOWED INACCURATE TEM...               4   
6020  NO DEPLOYMENT OF DRIVER'S AIRBAG DURING ACCIDE...               2   
6074  FALSE DEPLOYMENT OF DRIVER'S/PASSENGER'S AIRBA...               2   
6155                               ELECTRICAL PROBLEMS.               1   
6172  HOLE IN ALUMINUM WHEEL CAST, CAUSING LOSS OF A...               3   

     predicted_category  
22                    4  
65                    3  
154                  

In [80]:
sample_df.head()

,COMPDESC,CDESCR,Category_Label,predicted_category
22,AIR BAGS,"UPON IMPACT, DURING FRONTAL CRASH AT APPROX. 1...",2,4
65,WHEELS,RIGHT FRONT WHEEL LOCKED UP VEHICLE WENT INTO ...,3,3
154,AIR BAGS,DRIVER AND PASSENGER AIRBAG DID NOT DEPLOY IN ...,2,4
228,WHEELS,WHEEL CRACKED ON RIGHT FRONT WHEEL WHILE DRIVI...,3,3
310,WHEELS,WHEELS SKID WHEN BRAKING. *AK,3,3


In [82]:
from sklearn.metrics import classification_report

# Ensure all labels are strings and strip whitespace
sample_df['Category_Label'] = sample_df['Category_Label'].astype(str).str.strip()
sample_df['predicted_category'] = sample_df['predicted_category'].astype(str).str.strip()

# Generate classification report
report = classification_report(
    sample_df['Category_Label'],
    sample_df['predicted_category'],
    labels=[
        '1',
        '2',
        '3',
        '4',
        '5'
    ],
    zero_division=0
)

print(report)


              precision    recall  f1-score   support

           1       0.51      0.69      0.58      2150
           2       0.99      0.17      0.29      4836
           3       0.99      0.18      0.31      1259
           4       0.30      0.69      0.41       698
           5       0.16      0.66      0.26      1057

    accuracy                           0.37     10000
   macro avg       0.59      0.48      0.37     10000
weighted avg       0.75      0.37      0.36     10000

